# Assignment 2 - Metabolic Modeling (Week 2)

## Setup and Imports

In [41]:
%pip install cobra
%pip install panda
%pip install escher



Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/site-packages/pip/__main__.py", line 9, in <module>
    if sys.path[0] in ("", os.getcwd()):
FileNotFoundError: [Errno 2] No such file or directory
Note: you may need to restart the kernel to use updated packages.
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineToo

In [1]:
import cobra as cobra
import pandas as pd 
import escher



In [2]:
data = pd.read_csv("KEN3170_Assignment_2026_e_coli_core_expression.csv")
reaction_data = dict(zip(data["# Reaction ID"], data[" reaction activity [mmol/gDW/h] "]))
builder = escher.Builder(map_name="e_coli_core.Core metabolism", eaction_data=reaction_data)
builder
# you still need to load the data by yourself

Builder()

In [3]:
# after observing we can notice that "GAPD", "PGK", "PGM", "ENO", "PYK" form a linear pathway
linear_pathway = ["GAPD", "PGK", "PGM", "ENO", "PYK"]
# only include this path
comparison = data[data["# Reaction ID"].isin(linear_pathway)].copy()
# order such that the path has the correct values
comparison["# Reaction ID"] = pd.Categorical(comparison["# Reaction ID"],categories=linear_pathway,ordered=True)
comparison = comparison.sort_values("# Reaction ID")
comparison


,# Reaction ID,reaction activity [mmol/gDW/h]
46,GAPD,24.5
3,PGK,24.0
7,PGM,21.7
30,ENO,29.3
22,PYK,28.2


In [15]:
from cobra import Reaction
reaction_data["PGK"]

24.0

In [10]:
# get the activity values
activities = comparison[" reaction activity [mmol/gDW/h] "]
print("Are all activities equal?", activities.nunique() == 1)


Are all activities equal? False


### 1.a
As shown above, the **maximal reaction activities differ within the linear pathway**. This is due to the fact that they represent **capacity, but not the flux itself**. Therefore, unlike the steady-state fluxes observed in the interactive session, they are **not required to be equal due to mass balance constraints**.

### 1.b
The grey arrows show either **zero or no data (`nd`)**. Zero represents that the **maximal reaction activity for this reaction is zero**, whereas no data means that it **has not yet been determined in this model (missing information)**.


In [9]:
model = cobra.io.load_json_model('e_coli_core.json')
model.reactions.get_by_id("PFK")
# the model already defined weather or not a reaction is reversible based on the lower and upper bounds

Reaction identifier,PFK
Name,Phosphofructokinase
Memory address,0x12f042d30
Stoichiometry,"atp_c + f6p_c --> adp_c + fdp_c + h_c ATP C10H12N5O13P3 + D-Fructose 6-phosphate --> ADP C10H12N5O10P2 + D-Fructose 1,6-bisphosphate + H+"
GPR,b3916 or b1723
Lower bound,0.0
Upper bound,1000.0


# Task 2

In [30]:
def modify_bounds(reaction_data,model):

    # TO DO: For the glucose exchange reaction (“EX_glc__D_e”; see practical), please remove the pre-existing
    # maximal absolute flux bound and use the high absolute default bound instead
   

    # leave the lower flux as it is for ATPM
    atpm_lower_bound = model.reactions.get_by_id("ATPM").lower_bound

    for reaction in model.reactions:
        # special case
        if reaction.id == "EX_glc__D_e":
            continue
        # get all of the reactions that are both in reaction data and model
        if reaction.id in reaction_data:
            value = reaction_data[reaction.id]
            # print(value)
            # reversible reactions
            if reaction.lower_bound > 0 and reaction.upper_bound > 0:
                reaction.lower_bound = - value
                reaction.upper_bound = value
            else:
                # the lower bound is already zero
                reaction.upper_bound = value

    # does high absolute default bound means bounds of lb= -1000 and ub=1000
    # if yes then change - model.reactions.get_by_id("EX_glc__D_e").lower_bound = -1000

    model.reactions.get_by_id("ATPM").lower_bound = atpm_lower_bound
    
    return model
    


In [31]:
model = modify_bounds(reaction_data, model)
bounds_df = pd.DataFrame([{"Reaction": reaction.id,"Lower Bound": reaction.lower_bound,
"Upper Bound": reaction.upper_bound}for reaction in model.reactions])

bounds_df

,Reaction,Lower Bound,Upper Bound
0,PFK,0.0,13.1
1,PFL,0.0,0.0
2,PGI,-1000.0,11.1
3,PGK,-1000.0,24.0
4,PGL,0.0,7.3
...,...,...,...
90,NADH16,0.0,40.1
91,NADTRHD,0.0,1.3
92,NH4t,-1000.0,1000.0
93,O2t,-1000.0,1000.0
